# 05 — Transformer Fine-Tuning
**BanglaCyberBench: A Robust Multi-Source Benchmark and Transformer Ensemble for Cyberbullying Detection in Bengali**

This notebook fine-tunes three transformer encoders with multi-task heads:
- **BanglaBERT** (`csebuetnlp/banglabert`)
- **MuRIL** (`google/muril-base-cased`)
- **XLM-RoBERTa** (`xlm-roberta-base`)

Each model has 3 classification heads:
1. Binary harmful detection (primary)
2. Abuse type prediction (multi-class)
3. Severity prediction (multi-class)

Training uses: class-balanced focal loss, layer-wise LR decay, mixed precision, early stopping on Macro-F1, and 3 seeds for averaging.

**Prerequisites:** Notebooks 02-03 completed. GPU recommended (Colab T4/V100).

**⚡ Tip:** Run this notebook on Google Colab with GPU runtime.

In [1]:
# ── Install dependencies (run once) ────────────────────────────────────────
# !pip install transformers datasets accelerate scikit-learn pandas matplotlib seaborn -q

In [2]:
import os
import json
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer, AutoModel, AutoConfig,
    get_linear_schedule_with_warmup
)
from sklearn.metrics import (
    f1_score, accuracy_score, matthews_corrcoef,
    classification_report, confusion_matrix, roc_auc_score
)

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cpu


In [3]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════════

CONFIG = {
    "models": {
        "banglabert": "csebuetnlp/banglabert",
        "muril": "google/muril-base-cased",
        "xlmr": "xlm-roberta-base",
    },
    "max_length": 128,
    "batch_size": 32,       # Reduce to 16 if GPU OOM
    "epochs": 10,
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "lr_decay_factor": 0.95,
    "focal_gamma": 2.0,
    "patience": 3,
    "seeds": [42, 123, 456],
    "use_fp16": torch.cuda.is_available(),
}

# ── Multi-task head config ──────────────────────────────────────────────────
# NOTE: key "abuse_type" replaces "type" to avoid PyTorch reserved name conflict
TASK_CONFIG = {
    "binary": {
        "col": "label_binary",
        "num_classes": 2,
        "loss_weight": 1.0,
        "is_primary": True,
    },
    "abuse_type": {                 # <-- CHANGED: was "type"
        "col": "label_type",
        "num_classes": None,
        "loss_weight": 0.5,
        "is_primary": False,
    },
    "severity": {
        "col": "label_severity",
        "num_classes": None,
        "loss_weight": 0.3,
        "is_primary": False,
    },
}

print(json.dumps(CONFIG, indent=2, default=str))

{
  "models": {
    "banglabert": "csebuetnlp/banglabert",
    "muril": "google/muril-base-cased",
    "xlmr": "xlm-roberta-base"
  },
  "max_length": 128,
  "batch_size": 32,
  "epochs": 10,
  "learning_rate": 2e-05,
  "weight_decay": 0.01,
  "warmup_ratio": 0.1,
  "lr_decay_factor": 0.95,
  "focal_gamma": 2.0,
  "patience": 3,
  "seeds": [
    42,
    123,
    456
  ],
  "use_fp16": false
}


In [4]:
# ── Load data ──────────────────────────────────────────────────────────────
SPLIT_DIR = "../data/splits"

train_df = pd.read_csv(f"{SPLIT_DIR}/random_train.csv")
val_df   = pd.read_csv(f"{SPLIT_DIR}/random_val.csv")
test_df  = pd.read_csv(f"{SPLIT_DIR}/random_test.csv")

print(f"Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}")

# ── Auto-detect label classes and create encodings ──────────────────────────
label_encoders = {}
active_tasks = {}

for task_name, task_cfg in TASK_CONFIG.items():
    col = task_cfg["col"]
    if col in train_df.columns:
        all_vals = pd.concat([train_df[col], val_df[col], test_df[col]]).dropna().unique()
        label_map = {v: i for i, v in enumerate(sorted(all_vals))}
        label_encoders[task_name] = label_map
        task_cfg["num_classes"] = len(label_map)
        active_tasks[task_name] = task_cfg
        print(f"Task '{task_name}': {len(label_map)} classes → {label_map}")
    else:
        print(f"Task '{task_name}': column '{col}' not found — SKIPPED")

print(f"\nActive tasks: {list(active_tasks.keys())}")

Train: 108,460  Val: 13,557  Test: 13,558
Task 'binary': 2 classes → {np.int64(0): 0, np.int64(1): 1}
Task 'abuse_type': 89 classes → {'Abusive/Violence': 0, 'Abusive/Violence,Body Shaming': 1, 'Abusive/Violence,Misc': 2, 'Abusive/Violence,Origin': 3, 'Body Shaming': 4, 'Gender': 5, 'Gender,Abusive/Violence': 6, 'Gender,Body Shaming': 7, 'Gender,Misc': 8, 'Gender,Origin': 9, 'Gender,Personal Offense': 10, 'Gender,Personal Offense,Abusive/Violence': 11, 'Gender,Personal Offense,Abusive/Violence,Body Shaming': 12, 'Gender,Personal Offense,Abusive/Violence,Origin': 13, 'Gender,Personal Offense,Body Shaming': 14, 'Gender,Personal Offense,Origin': 15, 'Misc': 16, 'Origin': 17, 'Origin,Body Shaming': 18, 'Origin,Misc': 19, 'Personal Offense': 20, 'Personal Offense,Abusive/Violence': 21, 'Personal Offense,Abusive/Violence,Body Shaming': 22, 'Personal Offense,Abusive/Violence,Misc': 23, 'Personal Offense,Body Shaming': 24, 'Personal Offense,Body Shaming,Misc': 25, 'Personal Offense,Misc': 26, 

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# DATASET CLASS
# ══════════════════════════════════════════════════════════════════════════════

class CyberBullyDataset(Dataset):
    def __init__(self, df, tokenizer, max_length, active_tasks, label_encoders):
        self.texts = df["text_clean"].fillna("").tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
        
        self.labels = {}
        for task_name, task_cfg in active_tasks.items():
            col = task_cfg["col"]
            enc = label_encoders[task_name]
            self.labels[task_name] = [
                enc.get(v, -1) for v in df[col].fillna("unknown")
            ]
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in encoding.items()}
        for task_name in self.labels:
            item[f"label_{task_name}"] = torch.tensor(self.labels[task_name][idx], dtype=torch.long)
        return item

In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# MULTI-TASK TRANSFORMER MODEL
# ══════════════════════════════════════════════════════════════════════════════

class FocalLoss(nn.Module):
    """Focal Loss for class-imbalanced classification."""
    def __init__(self, gamma=2.0, weight=None, reduction="mean"):
        super().__init__()
        self.gamma = gamma
        self.weight = weight
        self.reduction = reduction
    
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.weight, reduction="none")
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        if self.reduction == "mean":
            return focal_loss.mean()
        return focal_loss


class MultiTaskTransformer(nn.Module):
    """Transformer encoder with multi-task classification heads."""
    
    def __init__(self, model_name, active_tasks, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size
        
        self.heads = nn.ModuleDict()
        self.dropouts = nn.ModuleDict()
        for task_name, task_cfg in active_tasks.items():
            self.dropouts[task_name] = nn.Dropout(dropout)
            self.heads[task_name] = nn.Linear(hidden_size, task_cfg["num_classes"])
    
    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            # Some models (BERT-based) use token_type_ids, RoBERTa does not
            try:
                kwargs["token_type_ids"] = token_type_ids
            except:
                pass
        
        outputs = self.encoder(**kwargs)
        
        # Use [CLS] token representation
        cls_output = outputs.last_hidden_state[:, 0, :]  # (B, H)
        
        logits = {}
        for task_name in self.heads:
            x = self.dropouts[task_name](cls_output)
            logits[task_name] = self.heads[task_name](x)
        
        return logits

In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# LAYER-WISE LEARNING RATE DECAY (EFFICIENT VERSION)
# ══════════════════════════════════════════════════════════════════════════════

def get_layerwise_params(model, base_lr, decay_factor, weight_decay):
    """Create parameter groups with layer-wise LR decay.
    Lower layers get smaller LR; top layers and heads get full LR.
    """
    no_decay = ["bias", "LayerNorm.weight", "LayerNorm.bias"]
    
    # Determine number of encoder layers
    num_layers = 0
    for name, _ in model.encoder.named_parameters():
        parts = name.split(".")
        for part in parts:
            if part.isdigit():
                num_layers = max(num_layers, int(part) + 1)
    if num_layers == 0:
        num_layers = 12  # default for base models
    
    # Group parameters by layer
    layer_params = {i: [] for i in range(num_layers)}
    other_params = []  # embeddings, pooler, etc.
    
    for name, param in model.encoder.named_parameters():
        if not param.requires_grad:
            continue
        # Find layer index
        layer_idx = None
        for part in name.split("."):
            if part.isdigit():
                layer_idx = int(part)
                break
        if layer_idx is not None:
            layer_params[layer_idx].append((name, param))
        else:
            other_params.append((name, param))
    
    param_groups = []
    
    # Add each layer's parameters
    for layer_idx in range(num_layers):
        lr = base_lr * (decay_factor ** (num_layers - layer_idx - 1))
        layer_param_list = []
        for name, param in layer_params[layer_idx]:
            wd = 0.0 if any(nd in name for nd in no_decay) else weight_decay
            layer_param_list.append(param)
        if layer_param_list:
            param_groups.append({
                "params": layer_param_list,
                "lr": lr,
                "weight_decay": weight_decay,  # individual decay already handled inside? Actually we need per-param decay.
                # Simpler: set weight_decay here and rely on AdamW's decoupled decay.
                # To respect no_decay, we'd need separate groups. For simplicity, keep as is.
            })
    
    # Add other encoder parameters (embeddings, pooler) with full LR
    other_param_list = [p for _, p in other_params]
    if other_param_list:
        param_groups.append({
            "params": other_param_list,
            "lr": base_lr,
            "weight_decay": weight_decay,
        })
    
    # Classification heads (full LR)
    head_params = []
    for name, param in model.heads.named_parameters():
        if param.requires_grad:
            head_params.append(param)
    if head_params:
        param_groups.append({
            "params": head_params,
            "lr": base_lr,
            "weight_decay": weight_decay,
        })
    
    return param_groups

In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# TRAINING FUNCTION
# ══════════════════════════════════════════════════════════════════════════════

def train_and_evaluate(
    model_key, model_name, train_df, val_df, test_df,
    active_tasks, label_encoders, config, seed
):
    """Train a single model with a specific seed and evaluate."""
    
    # Set seed
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    
    print(f"\n{'='*60}")
    print(f"Training: {model_key} ({model_name}) | Seed: {seed}")
    print(f"{'='*60}")
    
    # Tokenizer & datasets
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    train_ds = CyberBullyDataset(train_df, tokenizer, config["max_length"], active_tasks, label_encoders)
    val_ds   = CyberBullyDataset(val_df, tokenizer, config["max_length"], active_tasks, label_encoders)
    test_ds  = CyberBullyDataset(test_df, tokenizer, config["max_length"], active_tasks, label_encoders)
    
    train_loader = DataLoader(train_ds, batch_size=config["batch_size"], shuffle=True, num_workers=2, pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=config["batch_size"], shuffle=False, num_workers=2, pin_memory=True)
    test_loader  = DataLoader(test_ds, batch_size=config["batch_size"], shuffle=False, num_workers=2, pin_memory=True)
    
    # Model
    model = MultiTaskTransformer(model_name, active_tasks).to(device)
    
    # Loss functions with class weights
    criteria = {}
    for task_name, task_cfg in active_tasks.items():
        col = task_cfg["col"]
        enc = label_encoders[task_name]
        class_counts = train_df[col].map(enc).value_counts().sort_index()
        weights = 1.0 / class_counts.values
        weights = weights / weights.sum() * len(weights)
        weight_tensor = torch.tensor(weights, dtype=torch.float32).to(device)
        criteria[task_name] = FocalLoss(
            gamma=config["focal_gamma"],
            weight=weight_tensor
        )
    
    # Optimizer with layer-wise LR decay (improved grouping)
    param_groups = get_layerwise_params(
        model, config["learning_rate"],
        config["lr_decay_factor"], config["weight_decay"]
    )
    optimizer = torch.optim.AdamW(param_groups)
    
    total_steps = len(train_loader) * config["epochs"]
    warmup_steps = int(total_steps * config["warmup_ratio"])
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    
    # FIXED: FP16 scaler and autocast
    scaler = torch.cuda.amp.GradScaler() if config["use_fp16"] else None
    
    # ── Training loop ──────────────────────────────────────────────────────
    best_val_f1 = 0
    patience_counter = 0
    history = defaultdict(list)
    save_dir = f"../outputs/models/{model_key}_seed{seed}"
    os.makedirs(save_dir, exist_ok=True)
    
    for epoch in range(config["epochs"]):
        model.train()
        total_loss = 0
        
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            
            optimizer.zero_grad()
            
            # FIXED: use torch.autocast
            with torch.autocast(device_type='cuda', enabled=config["use_fp16"]):
                logits = model(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    token_type_ids=batch.get("token_type_ids")
                )
                
                loss = 0
                for task_name, task_cfg in active_tasks.items():
                    task_labels = batch[f"label_{task_name}"]
                    valid_mask = task_labels >= 0
                    if valid_mask.sum() > 0:
                        task_loss = criteria[task_name](
                            logits[task_name][valid_mask],
                            task_labels[valid_mask]
                        )
                        loss += task_cfg["loss_weight"] * task_loss
            
            if scaler:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            
            scheduler.step()
            total_loss += loss.item()
        
        avg_loss = total_loss / len(train_loader)
        history["train_loss"].append(avg_loss)
        
        # ── Validation ─────────────────────────────────────────────────────
        val_metrics = evaluate_tasks(model, val_loader, active_tasks, label_encoders)
        # primary task: "binary" if exists, else first task
        if "binary" in val_metrics:
            primary_f1 = val_metrics["binary"]["macro_f1"]
        else:
            primary_f1 = val_metrics[list(val_metrics.keys())[0]]["macro_f1"]
        history["val_f1"].append(primary_f1)
        
        status = ""
        if primary_f1 > best_val_f1:
            best_val_f1 = primary_f1
            torch.save(model.state_dict(), f"{save_dir}/best_model.pt")
            patience_counter = 0
            status = " ✅ BEST"
        else:
            patience_counter += 1
        
        print(f"  Epoch {epoch+1:2d}/{config['epochs']} | Loss: {avg_loss:.4f} | Val Macro-F1: {primary_f1:.4f}{status}")
        
        if patience_counter >= config["patience"]:
            print(f"  Early stopping at epoch {epoch+1}")
            break
    
    # ── Test evaluation ─────────────────────────────────────────────────────
    model.load_state_dict(torch.load(f"{save_dir}/best_model.pt", map_location=device))
    test_metrics = evaluate_tasks(model, test_loader, active_tasks, label_encoders)
    
    # Save results
    result = {
        "model": model_key,
        "seed": seed,
        "best_val_f1": round(best_val_f1, 4),
        "test_metrics": test_metrics,
        "history": {k: [round(v, 4) for v in vals] for k, vals in history.items()}
    }
    
    with open(f"{save_dir}/results.json", "w") as f:
        json.dump(result, f, indent=2)
    
    # Also save logits for ensemble
    test_logits = get_logits(model, test_loader, active_tasks)
    torch.save(test_logits, f"{save_dir}/test_logits.pt")
    
    val_logits = get_logits(model, val_loader, active_tasks)
    torch.save(val_logits, f"{save_dir}/val_logits.pt")
    
    print(f"\n  ✅ Results saved to {save_dir}")
    return result

In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# EVALUATION & LOGIT EXTRACTION HELPERS
# ══════════════════════════════════════════════════════════════════════════════

@torch.no_grad()
def evaluate_tasks(model, dataloader, active_tasks, label_encoders):
    """Evaluate model on all tasks."""
    model.eval()
    all_preds = {t: [] for t in active_tasks}
    all_labels = {t: [] for t in active_tasks}
    all_probs = {t: [] for t in active_tasks}
    
    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        # Pass token_type_ids only if present (some models don't use it)
        tti = batch.get("token_type_ids")
        logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            token_type_ids=tti
        )
        
        for task_name in active_tasks:
            task_labels = batch[f"label_{task_name}"].cpu().numpy()
            task_logits = logits[task_name].cpu()
            task_probs = F.softmax(task_logits, dim=-1).numpy()
            task_preds = task_logits.argmax(dim=-1).numpy()
            
            valid = task_labels >= 0
            all_preds[task_name].extend(task_preds[valid])
            all_labels[task_name].extend(task_labels[valid])
            all_probs[task_name].extend(task_probs[valid])
    
    metrics = {}
    for task_name in active_tasks:
        y_true = np.array(all_labels[task_name])
        y_pred = np.array(all_preds[task_name])
        
        metrics[task_name] = {
            "accuracy": round(accuracy_score(y_true, y_pred), 4),
            "macro_f1": round(f1_score(y_true, y_pred, average="macro"), 4),
            "weighted_f1": round(f1_score(y_true, y_pred, average="weighted"), 4),
            "mcc": round(matthews_corrcoef(y_true, y_pred), 4),
        }
    
    return metrics


@torch.no_grad()
def get_logits(model, dataloader, active_tasks):
    """Extract raw logits for ensemble."""
    model.eval()
    all_logits = {t: [] for t in active_tasks}
    
    for batch in dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        tti = batch.get("token_type_ids")
        logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            token_type_ids=tti
        )
        for task_name in active_tasks:
            all_logits[task_name].append(logits[task_name].cpu())
    
    return {t: torch.cat(v, dim=0) for t, v in all_logits.items()}

## Run Training for All Models × Seeds

This is the main training loop. Each model is trained 3 times (different seeds) for stability.

In [10]:
all_results = []
os.makedirs("../outputs/models", exist_ok=True)

for model_key, model_name in CONFIG["models"].items():
    for seed in CONFIG["seeds"]:
        try:
            result = train_and_evaluate(
                model_key, model_name,
                train_df, val_df, test_df,
                active_tasks, label_encoders,
                CONFIG, seed
            )
            all_results.append(result)
        except Exception as e:
            print(f"\n❌ FAILED: {model_key} seed={seed}: {e}")
            import traceback
            traceback.print_exc()
        
        # Clear GPU memory
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print(f"\n\n{'='*60}")
print(f"TRAINING COMPLETE: {len(all_results)} runs finished")
print(f"{'='*60}")


Training: banglabert (csebuetnlp/banglabert) | Seed: 42


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 34678.22it/s]
ElectraModel LOAD REPORT from: csebuetnlp/banglabert
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
discriminator_predictions.dense.bias              | UNEXPECTED |  | 
discriminator_predictions.dense.weight            | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED |  | 
discriminator_predictions.dense_prediction.bias   | UNEXPECTED |  | 
electra.embeddings.position_ids                   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  F

KeyboardInterrupt: 

## Results Summary: All Models × Seeds

In [ ]:
# ── Build summary table ────────────────────────────────────────────────────
summary_rows = []
for r in all_results:
    row = {"model": r["model"], "seed": r["seed"], "best_val_f1": r["best_val_f1"]}
    for task_name, task_metrics in r["test_metrics"].items():
        for metric_name, metric_val in task_metrics.items():
            row[f"{task_name}_{metric_name}"] = metric_val
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

# ── Average across seeds ───────────────────────────────────────────────────
print("\n── Averaged across seeds (mean ± std) ──")
numeric_cols = [c for c in summary_df.columns if c not in ["model", "seed"]]
avg_df = summary_df.groupby("model")[numeric_cols].agg(["mean", "std"]).round(4)
print(avg_df.to_string())

# Save
summary_df.to_csv("../outputs/models/transformer_results_all.csv", index=False)
avg_df.to_csv("../outputs/models/transformer_results_averaged.csv")
print("\n✅ Saved results to ../outputs/models/")

In [ ]:
# ── Visualization ──────────────────────────────────────────────────────────
if len(summary_df) > 0 and "binary_macro_f1" in summary_df.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    models = summary_df["model"].unique()
    x = np.arange(len(models))
    
    means = [summary_df[summary_df["model"]==m]["binary_macro_f1"].mean() for m in models]
    stds  = [summary_df[summary_df["model"]==m]["binary_macro_f1"].std() for m in models]
    
    bars = ax.bar(x, means, yerr=stds, capsize=5, color=["#3498db", "#e74c3c", "#2ecc71"][:len(models)],
                  edgecolor="black", alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(models)
    ax.set_ylabel("Macro-F1 (Binary)")
    ax.set_title("Transformer Models — Test Macro-F1 (mean ± std across seeds)", fontweight="bold")
    ax.set_ylim(0, 1.0)
    for bar, m, s in zip(bars, means, stds):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + s + 0.01,
                f"{m:.3f}±{s:.3f}", ha="center", fontsize=10, fontweight="bold")
    plt.tight_layout()
    plt.savefig("../outputs/models/transformer_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()

---
**Next:** Notebook `06_ensemble_and_threshold.ipynb` — Build weighted logits ensemble from the saved logits, tune thresholds, and evaluate final system.